# 02 · التنظيف والتوحيد
- تحويل "10:30pm" إلى وقت كامل بتوقيت الرياض (والعروض بعد منتصف الليل تُحسب للتاريخ الصحيح)
- توحيد أنواع الشاشات واللغات
- ربط نفس الفلم بين الدور والنسخ المدبلجة

In [ ]:
import sys, logging
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
import pandas as pd
pd.set_option("display.max_rows", 60)

In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo
from pipeline import clean, match

raw_path = ROOT / "data" / "raw_latest.pkl"
if raw_path.exists():
    raw = pd.read_pickle(raw_path)
else:  # لو ما شغّلت 01، نستخدم ملف الاختبار
    from pipeline.scrapers import vox
    fx = ROOT / "tests" / "fixtures" / "vox_showtimes.html"
    raw = pd.DataFrame([s.to_dict() for s in vox.parse_showtimes_html(fx.read_text(encoding="utf-8"), "riyadh-park-riyadh")])
now = datetime.now(ZoneInfo("Asia/Riyadh"))
print(len(raw), "صف خام")

In [ ]:
tidy = clean.clean(raw, now)
tidy = match.assign_movie_ids(tidy)
print(f"{len(raw)} خام ← {len(tidy)} بعد التنظيف ({len(raw)-len(tidy)} محذوف)")
tidy[["title_clean","cinema_name","experience","language","start","movie_id"]].head(20)

## فحص الجودة
أي صف هنا يحتاج انتباه.

In [ ]:
from zoneinfo import ZoneInfo
tz = ZoneInfo("Asia/Riyadh")
bdate = clean.business_date(now, 5)
unparsed = raw[raw["time_text"].map(lambda t: clean.parse_start(t, bdate, tz, 5) is None)]
print("أوقات ما انقرت:", unparsed["time_text"].unique().tolist() or "لا شي")
print("أنواع الشاشات:", tidy["experience"].value_counts().to_dict())
print("اللغات:", tidy["language"].value_counts().to_dict())


## توحيد الأفلام
كل `movie_id` لازم يكون فلم واحد. لو شفت فلمين مختلفين بنفس المعرّف، ارفع `threshold` في `match.assign_movie_ids`.

In [ ]:
(tidy.groupby("movie_id")["title_clean"].agg(lambda s: sorted(set(s)))
     .to_frame("الأسماء").assign(عدد=lambda d: d["الأسماء"].str.len())
     .sort_values("عدد", ascending=False))